## Import libraries 

In [1]:
from platform import python_version
print(python_version())

3.8.5


In [1]:
import os
import json
import pandas as pd
import pyodbc as ob
import numpy as np
from datetime import datetime, timedelta

## Import json 

In [2]:
os.listdir('./json/')

['llc_0028_vardict_data_v2.json']

In [3]:
with open('./json/llc_0028_vardict_data_v2.json', 'r') as f:
    project_vars = json.load(f)

To see structure of `study_vars`, see [`llc_0028_mkvardict_syntax_v1.ipynb`](llc_0028_mkvardict_syntax_v1.ipynb)

## Make directory to save data 

In [4]:
if 'data' not in os.listdir():
    os.mkdir('data')

## Open connection 

In [5]:
server = "SERPSQL"
database = "UKSERPUKLLC"
cnxn_str ="DRIVER={}; SERVER={}; DATABASE={}; Trusted_Connection=yes".format(
            "{SQL Server Native Client 11.0}",
            server,
            database
        )
con = ob.connect(cnxn_str)

## Functions 

In [6]:
def read_table(llc_proj, table, con):
    
    return pd.read_sql(f''' SELECT * FROM {llc_proj}.{table}''', con=con)

In [7]:
def get_covid_status(dta, dict_, study):
    
    var_ = dict_['var']
    
    cats2harmonised = dict_['harmonised']
    
    if study in ['sabre','nextstep','nhsd46','mcs','bcs70','ncds']:
        
        return [cats2harmonised[str(c)] if str(c) in cats2harmonised else c for c in dta[var_].values]
    else:
        return [cats2harmonised[c] if c in cats2harmonised else c for c in dta[var_].values]



In [8]:
def get_covid_date(dta, var_, study):
    
    #
    if type(var_)==dict:
        
        covid_date = []
        
        months = dta[var_['month']].values
        years = dta[var_['year']].values

        for i,v in enumerate(dta[var_['day']].values):

            if np.isnan(v):

                covid_date.append(v)

            else:

                covid_date.append(str(int(v)) + '/' + str(int(months[i])) + '/' + str(int(years[i])))
                
        return covid_date

    #twins cope4
    elif type(var_)==list:
        
        first_date = dta[var_[0]].values
        second_date = dta[var_[1]].values
        #keep date of second infection of exists
        covid_date = [v if '/' in v else first_date[i] for i,v in enumerate(second_date)]
        #strip whitespace and add trailing 0 where needed
        covid_date = [c.strip() if len(c.strip().split('/')[0])>1 else '0'+ c.strip() for c in covid_date]
        #for dates reported as m/y only, add d=15 (~midpoint)
        covid_date = [c if c.split('/')[0]!='0' else '15'+ c[1:] for c in covid_date]
        
        #at this point, covid_date either m/d/y, d/m/y, or missing
        #want to convert dates to d/m/y to match other studies
        
        parsed_covid_date = []
        for i,v in enumerate(covid_date):
            try:
                dt = datetime.strptime(v,'%m/%d/%Y')
                parsed_covid_date.append(datetime.strftime(dt,'%d/%m/%Y'))
            except ValueError:
                if '/' in v:
                    try:
                        dt = datetime.strptime(v,'%d/%m/%Y')
                        parsed_covid_date.append(datetime.strftime(dt,'%d/%m/%Y'))
                    except ValueError:
                        print(v)
                        parsed_covid_date.append('NA')
                else:
                    parsed_covid_date.append('NA')
        
        return parsed_covid_date
        
    else:
        
        covid_date = dta[var_].values
        
        if study=='bib':
            #BiB var_ free text - hacky code below to deal with it
            covid_date = [v if '/' in str(v) else 'NA' for v in covid_date]
            covid_date = [s if len(s.split('/')[-1])==4 and '/' in s else s+'20' for s in covid_date]
            covid_date = [np.nan if s in ['NA20','n/a20'] or 'n/a' in s else s for s in covid_date]
            
        elif study in ['sabre','nextstep','nhsd46','mcs','bcs70','ncds']:
            #CLS have integers representing month of infection
            
            date_dict = {1:'15/02/2020',
                         2:'15/03/2020',
                         3:'15/04/2020',
                         4:'15/05/2020',
                         5:'15/06/2020',
                         6:'15/07/2020',
                         7:'15/08/2020',
                         8:'15/09/2020',
                         9:'15/10/2020',
                         10:'15/11/2020',
                         11:'15/12/2020',
                         12:'15/01/2021',
                         13:'15/02/2021',
                         14:'15/03/2021',
                        }
            
            covid_date = [date_dict[c] if c in date_dict else c for c in covid_date]
        
        return covid_date     

In [9]:
def get_symptom_date(dta, var_):
    
    #alspac now dict
    if type(var_)==dict:
        
        symptom_date = []
        
        months = dta[var_['month']].values
        
        if type(var_['year'])==str: #ALSPAC
            
            years = dta[var_['year']].values
            
        else: #CLS
            
            years = [var_['year'] for i in range(len(months))]

        for i,v in enumerate(dta[var_['day']].values):

            if np.isnan(v):

                symptom_date.append(v)

            else:
                
                symptom_date.append(str(int(v)) + '/' + str(int(months[i])) + '/' + str(int(years[i])))
        
        #in ALSPAC and CLS, this is response date
        #asked about symptoms in 'last 2 weeks' subtract 1 week from derived symptom_date  
        
        symptom_date_adjusted = []
        
        for v in symptom_date:
            
            if type(v)!=str:
                
                symptom_date_adjusted.append(v)
                
            else:
                try:
                    t = datetime.strptime(v,'%d/%m/%Y') - timedelta(days=7)
                    symptom_date_adjusted.append(datetime.strftime(t,'%d/%m/%Y'))
                except ValueError:
                    symptom_date_adjusted.append(np.nan)
        
        return symptom_date_adjusted
    
    
    elif '/' in var_:
        
        return [var_ for i in range(dta.shape[0])]
    
    else:
        
        return dta[var_].values

In [10]:
def get_symptoms(dta, study_id, dict_):
    
    columns = list(dict_.keys())
    
    columns.append(study_id)
    
    symptoms = dta[columns]
    
    symptoms = symptoms.rename(columns = dict_)
    
    return symptoms

In [11]:
def derive_time_since_covid(df,study):
    
    df_copy = df.copy()
    
    time = []

    for i,v in enumerate(df.covid_date.values):
    
        if type(v)!=str:

            time.append(v)

        else:

            try: 
                x = (datetime.strptime(df.symptom_date.values[i],'%d/%m/%Y')-datetime.strptime(v,'%d/%m/%Y')).days/7
                    
                time.append(x)

            except ValueError:

                time.append(np.nan)
                #print(v)
            except TypeError:
            
                time.append(np.nan)
                #print(v)
    df_copy['time_since_covid'] = time
    
    return df_copy

In [12]:
def derive_covid_status(df):
    
    df_copy = df.copy()
    
    status = []

    for v in df.time_since_covid.values:

        # if time since covid is 'NA', participant didnt have covid
        # status = 0
        if np.isnan(v):

            status.append(0)

        elif v<0:
            
            status.append(0)

        #if time since covid < 12 weeks, status = 1
        elif 0<=v<12:

            status.append(1)

        #if time since covid > 12 weeks, status = 2
        elif v>=12:

            status.append(2)

        #if none of above, some sort of problem that needs checking
        else:

            status.append('NA')

    df_copy['covid_status'] = status
    
    return df_copy

In [13]:
def get_func_limitation(dta, study_id, var_ ):
    
    columns = [study_id, var_]
    
    fl = dta[columns]
    
    fl = fl.rename(columns = {var_:'functional_limitation'})
    
    return fl

In [14]:
def extract_data(llc_proj, study, project_vars, con):
    
    dta_dir = 'S:/LLC_0028/data/processed_study_data'
    
    study_vars = project_vars[study]
    
    table = study_vars['table']
    
    dta = read_table(llc_proj, table, con)
    
    study_id = f'{llc_proj}_stud_id'
    
    extracted = pd.DataFrame()
    
    extracted[study_id] = dta[study_id].values
    
    extracted['covid_yes_no'] = get_covid_status(dta, study_vars['covid_yes_no'], study)
    
    extracted['covid_date'] = get_covid_date(dta, study_vars['covid_date'], study)
    
    extracted['symptom_date'] = get_symptom_date(dta, study_vars['symptom_date'])
    
    extracted = extracted.loc[extracted.covid_date != 'Aug/Sept']
    
    symptoms = get_symptoms(dta, study_id, study_vars['symptoms'])
    
    # in alspac, 'No' recorded as 2 - replace with 0 to match other studies
    # in CLS, memory and concentrating 1 or 2 = 0, 3 or 4 = 1
    symptoms = symptoms.replace(2,0)
    symptoms = symptoms.replace(3,1)
    symptoms = symptoms.replace(4,1)
    
    
    final = extracted.merge(symptoms, left_on = study_id, right_on = study_id, how='left')
    
    final = derive_time_since_covid(final, study)
    
    final = derive_covid_status(final)
    
    if 'functional_limitation' in study_vars:
        
        fl = get_func_limitation(dta, study_id, study_vars['functional_limitation'])
        
        final = final.merge(fl, left_on = study_id, right_on = study_id, how='left')
    
    study = ('').join(study.split('_'))
    
    final.to_csv(dta_dir + f'/{llc_proj.lower()}_{study}dat_data_v1.csv')
    
    print(f'{study} complete')
    
    return final

## Extract data 

In [15]:
llc_proj = 'LLC_0028'

In [16]:
%pdb

Automatic pdb calling has been turned ON


In [17]:
%%time

for study in project_vars:
    extract_data(llc_proj, study, project_vars, con)

alspacm complete
alspacy complete
bib complete
track19 complete
15/01/
twins complete
sabre complete
nhsd46 complete
nextstep complete
mcs complete
bcs70 complete
ncds complete
Wall time: 25.4 s
